# 06. Baseline models

Phase 3, task 3.3. The simple forecasts that any model has to beat, scored with the evaluation framework of task 3.1 (walk-forward blocks with a gap equal to the lead time) on the features of task 3.2. These are the first runs of the project that count, and they are logged with MLflow.

**Target.** The daily mean PM2.5 of day O + lead, forecast at the end of day O, for lead 1, 2 and 3 days. US station only.

**Baselines**

| Name | What it does |
|---|---|
| `persistence` | tomorrow equals today |
| `mean_7d` | the mean of the last 7 days |
| `climatology` | the mean of training days within 15 days of the same day of the year |
| `seasonal_naive` | the same date one year earlier (needs a year of history, so it covers fewer rows) |
| `ridge_history` | linear regression (ridge) on PM2.5 history and calendar |
| `ridge_forecast_weather` | the same plus weather observed up to today and the archived forecast for the target day |
| `ridge_forecast_no_wind` | the same without the forecast wind (the biased variable) |

The ridge penalty is chosen on the last year of the training window. Missing predictors are filled with training-period medians plus missing-value flags, and predictions are never below zero.

**Two test views**

- **Observed weather (upper bound).** Weather models get the weather that actually happened on the target day, as if the forecast were perfect. This covers all winters from 2020-21 to 2024-25.
- **Real forecasts.** Weather models get the archived forecast issued before the origin. These exist from January 2024, so only winter 2023-24 (from 20 January), the months March to October 2024 and winter 2024-25 are scored, and every model is scored on exactly the same days.

Models are always trained with the observed weather standing in for the forecast, because there is too little forecast history to train on. Winter means November to February.

In [1]:
import warnings
from datetime import date
from pathlib import Path

import pandas as pd

from aeron.evaluation import report, splits
from aeron.evaluation.runner import common_rows, run_models
from aeron.models.baselines import default_models

warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

daily = pd.read_csv(
    ROOT / "data" / "processed" / "daily_us_embassy.csv", index_col=0, parse_dates=True
)
blocks = splits.seasonal_blocks(2020, date(2025, 2, 25))
LEADS = (1, 2, 3)
VIEWS = ("observed", "forecast")
MAIN = [
    "persistence",
    "mean_7d",
    "climatology",
    "ridge_history",
    "ridge_forecast_weather",
    "ridge_forecast_no_wind",
]
ALL = [*MAIN, "seasonal_naive"]

results, fitted = {}, {}
for view in VIEWS:
    for lead in LEADS:
        models = default_models()
        results[(view, lead)] = run_models(daily, models, lead, blocks, view)
        fitted[(view, lead)] = models
print(f"{len(blocks)} blocks, from {blocks[0].name} to {blocks[-1].name}")
print("scored rows (all models available) by view and lead:")
print({(v, l): len(common_rows(results[(v, l)], MAIN)) for v in VIEWS for l in LEADS})

9 blocks, from winter 2020-21 to winter 2024-25
scored rows (all models available) by view and lead:
{('observed', 1): 1126, ('observed', 2): 1101, ('observed', 3): 1088, ('forecast', 1): 357, ('forecast', 2): 352, ('forecast', 3): 349}


## 1. Logging the runs

One MLflow run per view, lead and model, recording the settings, the split scheme, the data and code versions, the metrics for winter, other months and all rows, and the predictions. Skipped if MLflow is not installed (`pip install -e ".[tracking]"`).

In [2]:
try:
    from aeron.evaluation.tracking import tracked_run
except ImportError:
    tracked_run = None
    print("MLflow not installed, runs are not logged")

LOG_RUNS = tracked_run is not None
if LOG_RUNS:
    logged = 0
    for (view, lead), pred in results.items():
        scored = common_rows(pred, MAIN)
        summary = report.summarize(scored, ALL[:-1])
        for model in fitted[(view, lead)]:
            if model.name == "seasonal_naive":
                rows = common_rows(pred, ALL)
                summary_model = report.summarize(rows, ["seasonal_naive"])
            else:
                rows, summary_model = scored, summary[summary["model"] == model.name]
            features = list(getattr(model, "_columns", []))
            params = {
                "model": model.name,
                "lead_days": lead,
                "test_view": view,
                "alpha": getattr(model, "chosen_alpha", None),
                "threshold": report.EVENT_THRESHOLD,
            }
            split = {
                "scheme": "walk-forward, expanding window",
                "first_block": blocks[0].name,
                "last_block": blocks[-1].name,
                "gap_days": lead,
            }
            with tracked_run(
                f"baseline-{view}-lead{lead}-{model.name}",
                params,
                seed=0,
                split=split,
                features=features,
                tags={"task": "3.3 baselines"},
            ) as run:
                for _, row in summary_model.iterrows():
                    run.log_metrics(
                        {
                            k: row[k]
                            for k in (
                                "n",
                                "mae",
                                "rmse",
                                "bias",
                                "r2",
                                "precision",
                                "recall",
                                "f1",
                                "always_yes_f1",
                            )
                        },
                        prefix=f"{row['group']}",
                    )
                run.log_predictions(
                    rows[["target_date", "block", "season", "y", model.name]]
                )
            logged += 1
    print(f"logged {logged} runs to mlruns/")

logged 42 runs to mlruns/


## 2. Observed weather (upper bound for the weather models), all winters

MAE in ug/m3 on the rows where every main model has a prediction. `winter n` is the number of scored winter days.

In [3]:
def mae_table(view, group):
    cols = {}
    for lead in LEADS:
        scored = common_rows(results[(view, lead)], MAIN)
        s = report.summarize(scored, MAIN)
        s = s[s["group"] == group].set_index("model")
        cols[(f"lead {lead}", "MAE")] = s["mae"]
        cols[(f"lead {lead}", "RMSE")] = s["rmse"]
        cols[(f"lead {lead}", "n")] = s["n"]
    return pd.DataFrame(cols).round(1)


print("winter days")
mae_table("observed", "winter")

winter days


lead 1            lead 2            lead 3           
                          MAE  RMSE    n    MAE  RMSE    n    MAE  RMSE    n
model                                                                       
persistence              21.3  29.9  441   27.8  41.7  430   30.2  43.5  419
mean_7d                  22.8  30.7  441   24.4  35.3  430   25.5  36.4  419
climatology              26.0  35.6  441   26.0  37.5  430   25.7  37.2  419
ridge_history            17.0  23.0  441   22.6  33.1  430   23.8  34.4  419
ridge_forecast_weather   15.2  20.6  441   18.7  29.1  430   19.6  30.2  419
ridge_forecast_no_wind   15.7  21.3  441   19.9  30.2  430   20.8  31.3  419

In [4]:
print("March to October days")
mae_table("observed", "other")

March to October days


lead 1            lead 2            lead 3           
                          MAE  RMSE    n    MAE  RMSE    n    MAE  RMSE    n
model                                                                       
persistence               7.3  11.3  685    9.1  13.6  671   10.3  16.0  669
mean_7d                   7.9  11.9  685    8.3  12.1  671    8.8  13.0  669
climatology              12.1  15.6  685   12.1  15.0  671   12.2  15.2  669
ridge_history             6.4   9.6  685    8.1  10.9  671    8.7  11.8  669
ridge_forecast_weather    6.7   9.7  685    8.6  11.3  671    9.4  12.2  669
ridge_forecast_no_wind    6.7   9.6  685    8.8  11.4  671    9.5  12.4  669

In [5]:
print("all scored days")
mae_table("observed", "all")

all scored days


lead 1             lead 2             lead 3            
                          MAE  RMSE     n    MAE  RMSE     n    MAE  RMSE     n
model                                                                          
persistence              12.8  20.7  1126   16.4  28.1  1101   18.0  29.8  1088
mean_7d                  13.7  21.3  1126   14.6  24.0  1101   15.2  24.8  1088
climatology              17.6  25.4  1126   17.5  26.2  1101   17.4  26.0  1088
ridge_history            10.6  16.2  1126   13.7  22.4  1101   14.5  23.2  1088
ridge_forecast_weather   10.0  14.9  1126   12.5  20.2  1101   13.4  21.0  1088
ridge_forecast_no_wind   10.2  15.3  1126   13.1  20.9  1101   13.9  21.7  1088

In [6]:
# Paired differences in MAE for winter (negative: the first model is better), block bootstrap 95% intervals.
pairs = [
    ("ridge_history", "persistence"),
    ("ridge_history", "climatology"),
    ("ridge_forecast_weather", "ridge_history"),
    ("ridge_forecast_weather", "persistence"),
    ("ridge_forecast_weather", "climatology"),
    ("ridge_forecast_no_wind", "ridge_forecast_weather"),
]


def difference_table(view, group="winter"):
    rows = []
    for a, b in pairs:
        row = {"comparison": f"{a} minus {b}"}
        for lead in LEADS:
            scored = common_rows(results[(view, lead)], MAIN)
            d = report.paired_difference(scored, a, b, group)
            row[f"lead {lead}"] = f"{d['diff']:+.1f} [{d['lo']:+.1f}, {d['hi']:+.1f}]"
        rows.append(row)
    return pd.DataFrame(rows).set_index("comparison")


difference_table("observed")

,lead 1,lead 2,lead 3
comparison,,,
ridge_history minus persistence,"-4.3 [-5.7, -3.0]","-5.2 [-7.5, -3.1]","-6.3 [-8.9, -3.9]"
ridge_history minus climatology,"-9.0 [-11.5, -6.7]","-3.4 [-5.3, -1.7]","-1.8 [-3.7, -0.2]"
ridge_forecast_weather minus ridge_history,"-1.8 [-2.5, -1.2]","-3.9 [-4.8, -3.0]","-4.2 [-5.4, -2.9]"
ridge_forecast_weather minus persistence,"-6.1 [-7.5, -4.9]","-9.1 [-11.4, -6.9]","-10.5 [-13.3, -7.9]"
ridge_forecast_weather minus climatology,"-10.8 [-13.5, -8.4]","-7.3 [-9.2, -5.6]","-6.0 [-8.1, -4.3]"
ridge_forecast_no_wind minus ridge_forecast_weather,"+0.5 [+0.1, +0.9]","+1.2 [+0.7, +1.6]","+1.1 [+0.7, +1.6]"


In [7]:
# MAE per test block for the main baselines (lead 1 and lead 3).
for lead in (1, 3):
    scored = common_rows(results[("observed", lead)], MAIN)
    print(f"lead {lead}")
    print(
        report.per_block_mae(
            scored,
            ["persistence", "climatology", "ridge_history", "ridge_forecast_weather"],
        )
        .round(1)
        .to_string()
    )
    print()

lead 1
                days  persistence  climatology  ridge_history  ridge_forecast_weather
block                                                                                
winter 2020-21    85         14.2         19.9           12.6                    11.0
mar-oct 2021     168          7.3          8.4            7.0                     6.4
winter 2021-22    99         17.3         18.2           13.0                    11.2
mar-oct 2022      98          5.8          6.5            4.7                     6.2
winter 2022-23    68         22.0         38.2           19.6                    16.7
mar-oct 2023     205          6.9         14.3            5.7                     6.5
winter 2023-24    84         26.6         30.4           20.1                    19.1
mar-oct 2024     214          8.4         15.7            7.3                     7.4
winter 2024-25   105         26.0         26.8           20.2                    18.1

lead 3
                days  persistence  clim

## 3. Real archived forecasts, 2024 and 2025 only

Every model is scored on the same days, which all have a real forecast.

In [8]:
print("winter days")
mae_table("forecast", "winter")

winter days


lead 1            lead 2            lead 3           
                          MAE  RMSE    n    MAE  RMSE    n    MAE  RMSE    n
model                                                                       
persistence              24.3  31.2  143   32.8  42.0  140   35.6  44.9  137
mean_7d                  26.4  33.3  143   27.9  35.4  140   28.7  35.8  137
climatology              25.9  32.2  143   25.7  32.0  140   25.6  32.0  137
ridge_history            19.0  24.4  143   25.8  32.1  140   26.7  33.2  137
ridge_forecast_weather   18.2  23.4  143   22.4  28.1  140   23.5  28.8  137
ridge_forecast_no_wind   17.6  23.0  143   22.5  28.8  140   23.7  29.8  137

In [9]:
print("all scored days (winter and March to October 2024)")
mae_table("forecast", "all")

all scored days (winter and March to October 2024)


lead 1            lead 2            lead 3           
                          MAE  RMSE    n    MAE  RMSE    n    MAE  RMSE    n
model                                                                       
persistence              14.8  22.7  357   18.7  29.3  352   20.6  32.0  349
mean_7d                  16.1  24.2  357   16.7  25.0  352   17.2  25.6  349
climatology              19.8  25.4  357   19.4  24.4  352   19.3  24.2  349
ridge_history            12.0  18.1  357   15.5  22.5  352   16.2  23.1  349
ridge_forecast_weather   13.4  18.2  357   16.7  21.5  352   17.2  21.9  349
ridge_forecast_no_wind   11.6  17.3  357   14.2  20.5  352   14.9  21.2  349

In [10]:
difference_table("forecast")

,lead 1,lead 2,lead 3
comparison,,,
ridge_history minus persistence,"-5.3 [-7.1, -3.5]","-6.9 [-8.8, -5.0]","-8.8 [-11.8, -6.1]"
ridge_history minus climatology,"-6.9 [-8.6, -5.0]","+0.2 [-1.6, +1.8]","+1.2 [-1.0, +2.9]"
ridge_forecast_weather minus ridge_history,"-0.8 [-1.5, -0.2]","-3.4 [-4.8, -2.1]","-3.3 [-5.0, -1.5]"
ridge_forecast_weather minus persistence,"-6.1 [-7.8, -4.3]","-10.4 [-13.1, -7.6]","-12.1 [-16.4, -8.2]"
ridge_forecast_weather minus climatology,"-7.7 [-9.5, -5.8]","-3.3 [-5.0, -1.7]","-2.1 [-4.3, -0.0]"
ridge_forecast_no_wind minus ridge_forecast_weather,"-0.6 [-1.0, -0.1]","+0.1 [-0.7, +1.0]","+0.2 [-0.8, +1.2]"


## 4. Exceedance of 55.5 ug/m3 (daily mean)

A forecast value at or above 55.5 counts as a predicted exceedance. The F1 score is shown next to the F1 of always predicting an exceedance, which is the bar a useful forecast has to clear.

In [11]:
def event_table(view, group="winter"):
    cols = {}
    for lead in LEADS:
        scored = common_rows(results[(view, lead)], MAIN)
        s = report.summarize(scored, MAIN)
        s = s[s["group"] == group].set_index("model")
        cols[(f"lead {lead}", "F1")] = s["f1"]
        cols[(f"lead {lead}", "recall")] = s["recall"]
        cols[(f"lead {lead}", "precision")] = s["precision"]
        always = s["always_yes_f1"].iloc[0]
        cols[(f"lead {lead}", "always-yes F1")] = pd.Series(always, index=s.index)
        cols[(f"lead {lead}", "base rate")] = s["base_rate"]
    return pd.DataFrame(cols).round(2)


print("observed weather view, winter")
display(
    event_table("observed")[
        [
            c
            for c in event_table("observed").columns
            if c[1] in ("F1", "always-yes F1", "base rate")
        ]
    ]
)
print("real forecast view, winter")
display(
    event_table("forecast")[
        [
            c
            for c in event_table("forecast").columns
            if c[1] in ("F1", "always-yes F1", "base rate")
        ]
    ]
)

observed weather view, winter


lead 1                         lead 2                         lead 3                        
                           F1 always-yes F1 base rate     F1 always-yes F1 base rate     F1 always-yes F1 base rate
model                                                                                                              
persistence              0.70          0.62      0.45   0.59          0.62      0.44   0.56          0.61      0.44
mean_7d                  0.64          0.62      0.45   0.58          0.62      0.44   0.57          0.61      0.44
climatology              0.45          0.62      0.45   0.45          0.62      0.44   0.46          0.61      0.44
ridge_history            0.73          0.62      0.45   0.61          0.62      0.44   0.50          0.61      0.44
ridge_forecast_weather   0.76          0.62      0.45   0.74          0.62      0.44   0.71          0.61      0.44
ridge_forecast_no_wind   0.76          0.62      0.45   0.73          0.62      0.44   0.68          0.61      0.44

real forecast view, winter


lead 1                         lead 2                         lead 3                        
                           F1 always-yes F1 base rate     F1 always-yes F1 base rate     F1 always-yes F1 base rate
model                                                                                                              
persistence              0.66          0.63      0.46   0.53          0.63      0.46   0.49          0.64      0.47
mean_7d                  0.56          0.63      0.46   0.50          0.63      0.46   0.52          0.64      0.47
climatology              0.56          0.63      0.46   0.57          0.63      0.46   0.59          0.64      0.47
ridge_history            0.72          0.63      0.46   0.54          0.63      0.46   0.47          0.64      0.47
ridge_forecast_weather   0.76          0.63      0.46   0.74          0.63      0.46   0.71          0.64      0.47
ridge_forecast_no_wind   0.73          0.63      0.46   0.64          0.63      0.46   0.64          0.64      0.47

## 5. Seasonal naive (same date last year)

It needs a full year of history and the value of that date, so it is scored on its own, smaller set of rows, with persistence and climatology on the same rows.

In [12]:
rows = []
for lead in LEADS:
    scored = common_rows(results[("observed", lead)], ALL)
    s = report.summarize(scored, ["seasonal_naive", "persistence", "climatology"])
    s = s[s["group"] == "winter"].set_index("model")
    rows.append(
        {
            "lead": lead,
            "winter days": int(s["n"].iloc[0]),
            **{
                m: round(s.loc[m, "mae"], 1)
                for m in ("seasonal_naive", "persistence", "climatology")
            },
        }
    )
pd.DataFrame(rows).set_index("lead")

,winter days,seasonal_naive,persistence,climatology
lead,,,,
1,360,38.6,22.1,26.3
2,351,38.3,28.1,25.5
3,343,37.9,30.4,25.1


## 6. Saved metrics and chosen penalties

In [13]:
out = []
for (view, lead), pred in results.items():
    s = report.summarize(common_rows(pred, MAIN), MAIN)
    s.insert(0, "lead", lead)
    s.insert(0, "view", view)
    out.append(s)
    sn = report.summarize(common_rows(pred, ALL), ["seasonal_naive"])
    sn.insert(0, "lead", lead)
    sn.insert(0, "view", view + " (rows with last year's value)")
    out.append(sn)
table = pd.concat(out, ignore_index=True)
target = ROOT / "reports" / "model_evaluation"
target.mkdir(parents=True, exist_ok=True)
table.round(4).to_csv(target / "baseline_metrics.csv", index=False)
print(f"saved reports/model_evaluation/baseline_metrics.csv ({len(table)} rows)")
print("ridge penalty chosen per block (observed view, lead 1):")
print(
    {
        m.name: m.chosen_alpha
        for m in fitted[("observed", 1)]
        if hasattr(m, "chosen_alpha")
    }
)

saved reports/model_evaluation/baseline_metrics.csv (126 rows)
ridge penalty chosen per block (observed view, lead 1):
{'ridge_history': 10.0, 'ridge_forecast_weather': 100.0, 'ridge_forecast_no_wind': 100.0}


## 7. Findings

All numbers come from the cells above (US station, ridge penalties chosen on the last training year, 42 runs logged to MLflow). Errors are MAE in ug/m3 of the daily mean. These are the bars the tree models of task 3.4 have to clear.

**Simple forecasts are weak after one day.** In winter with observed weather (441, 430 and 419 scored days at 1, 2 and 3 days), persistence has MAE 21.3, 27.8 and 30.2, the 7-day mean 22.8, 24.4 and 25.5, and the day-of-year climatology 26.0, 26.0 and 25.7. Persistence is best of the three at one day and worst at three. The same date one year earlier is worse than all of them (38.6, 38.3 and 37.9 against 22.1, 28.1 and 30.4 for persistence on the same days) and is dropped as a serious baseline.

**A linear model on the series' own history already beats them.** Ridge regression on PM2.5 history and calendar has winter MAE 17.0, 22.6 and 23.8. Against persistence the difference is -4.3 [-5.7, -3.0], -5.2 [-7.5, -3.1] and -6.3 [-8.9, -3.9] ug/m3, and against climatology -9.0, -3.4 and -1.8 (the last interval [-3.7, -0.2] barely excludes zero). The intervals are paired block-bootstrap 95% intervals, unadjusted.

**Weather adds a clear amount in winter, and more at longer leads.** With the weather that happened (a perfect forecast, so an upper bound) ridge reaches 15.2, 18.7 and 19.6, which is 1.8, 3.9 and 4.2 ug/m3 better than the history-only ridge (intervals [-2.5, -1.2], [-4.8, -3.0] and [-5.4, -2.9]). With the real archived forecast (2024 and 2025, 143, 140 and 137 winter days) it is 18.2, 22.4 and 23.5 against 19.0, 25.8 and 26.7 for history only: better by 0.8 [-1.5, -0.2], 3.4 [-4.8, -2.1] and 3.3 [-5.0, -1.5]. On those same days persistence has 24.3, 32.8 and 35.6 and climatology 25.9, 25.7 and 25.6. So with real forecasts the weather model beats climatology by 7.7 [-9.5, -5.8], 3.3 [-5.0, -1.7] and 2.1 [-4.3, -0.0] ug/m3, which at three days is a small margin. Without weather the history-only model is no better than climatology at two and three days on these days (+0.2 and +1.2, intervals include zero).

**It does not help outside winter.** From March to October the history-only ridge is best (6.4, 8.1 and 8.7), slightly ahead of the weather ridge (6.7, 8.6 and 9.4) and of persistence (7.3, 9.1 and 10.3), with observed weather. On the real-forecast days (winter plus March to October 2024), the weather model without forecast wind has the lowest error overall (11.6, 14.2 and 14.9). Weather features are mainly a winter tool in this data.

**Forecast wind.** With perfect weather, dropping wind makes the model worse by 0.5, 1.2 and 1.1 ug/m3. With the real, biased forecast wind, dropping it helps at one day (-0.6 [-1.0, -0.1] in winter) and makes no measurable difference at two and three days (+0.1 and +0.2, intervals include zero). This agrees with the earlier gradient boosting experiment, and the bias correction of task 3.2 is the next thing to test.

**Exceedance of 55.5 ug/m3 in winter** (F1 against always predicting an exceedance, 0.62 to 0.64). With real forecasts the weather ridge scores 0.76, 0.74 and 0.71, the no-wind version 0.73, 0.64 and 0.64, history only 0.72, 0.54 and 0.47, persistence 0.66, 0.53 and 0.49, and climatology 0.56, 0.57 and 0.59. Only the weather ridge stays clearly above always-yes at all three leads. With observed weather the weather ridge scores 0.76, 0.74 and 0.71.

**Every winter.** At lead 1 and lead 3 the weather ridge has a lower MAE than persistence and climatology in each of the five winters (for example 18.1 against 26.0 and 26.8 in 2024-25 at one day). Winter 2024-25 at three days is the hardest case for persistence (39.6).

**Caveats.** One station. The real-forecast numbers rest on about 140 winter days from two winters. The weather ridge is trained with observed weather standing in for the forecast, which is a mismatch that the real-forecast view exposes. These are linear models with a fixed feature set and no tuning of the features. Results are not directly comparable to the earlier gradient boosting experiments because the rows and the split gap differ.

**What the tree models must beat (winter MAE)**

| View | Lead 1 | Lead 2 | Lead 3 |
|---|---|---|---|
| Real forecasts, best baseline (weather ridge) | 18.2 | 22.4 | 23.5 |
| Real forecasts, climatology | 25.9 | 25.7 | 25.6 |
| Observed weather, weather ridge | 15.2 | 18.7 | 19.6 |
